# Stage 3 — Honest Validation (rebuilt: no coordinates, with missingness)

Two problems to solve now:
1. **Coordinates are gone**, so we can't hold out geographic regions. We substitute **feature-space clustering** — group locations with similar spectral signatures, hold whole groups out. A weaker proxy for "unseen conditions" than geography, but far better than random folds.
2. **Train is clean, test is ~60% missing.** If we validate on clean data we'll be wildly optimistic. So we **corrupt the validation fold** to mimic the test, giving a realistic estimate.

This stage produces the number we actually trust.

In [1]:
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# new downloaded filenames (double underscores from the browser) — do NOT rename the files
TRAIN_PATH  = "Train.csv"
TEST_PATH   = "Test.csv"
SAMPLE_PATH = "SampleSubmission.csv"
NODATA = -9999            # satellite no-data sentinel in the new test set

In [2]:
# ============================================================
# Shared feature engineering — identical for train & test.
# Handles the -9999 no-data values via NaN-aware statistics.
# ============================================================
MONTHS = [f"{i:02d}" for i in range(1, 13)]
BANDS  = ["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]

def load_raw(path):
    """Load a CSV and convert the -9999 no-data sentinel to NaN."""
    return pd.read_csv(path).replace(NODATA, np.nan)

def safe_ratio(a, b):
    # NaN propagates naturally; +0 denominator -> inf, cleaned later
    return (a - b) / (a + b)

def build_features(df):
    """Raw monthly bands -> engineered features.
    NaN-aware: every summary is computed over the VALID months only.
    Adds *_nvalid = how many months actually had data (a quality signal)."""
    feats = {}; idx = {}
    for m in MONTHS:
        g = df[f"green_{m}"]; n = df[f"nir_{m}"]; r = df[f"red_{m}"]; s1 = df[f"swir1_{m}"]
        idx.setdefault("NDWI",  []).append(safe_ratio(g,  n))
        idx.setdefault("MNDWI", []).append(safe_ratio(g,  s1))
        idx.setdefault("NDVI",  []).append(safe_ratio(n,  r))
        idx.setdefault("NDMI",  []).append(safe_ratio(n,  s1))
        idx.setdefault("VHmVV", []).append(df[f"VH_{m}"] - df[f"VV_{m}"])
    def summarize(monthly, name):
        a = np.column_stack([np.asarray(x, float) for x in monthly])  # rows x 12
        feats[f"{name}_mean"]  = np.nanmean(a, 1)
        feats[f"{name}_std"]   = np.nanstd(a, 1)
        feats[f"{name}_min"]   = np.nanmin(a, 1)
        feats[f"{name}_max"]   = np.nanmax(a, 1)
        feats[f"{name}_range"] = np.nanmax(a, 1) - np.nanmin(a, 1)
        feats[f"{name}_med"]   = np.nanmedian(a, 1)
        feats[f"{name}_nvalid"]= np.isfinite(a).sum(1)     # data-quality feature
    for b in BANDS: summarize([df[f"{b}_{m}"] for m in MONTHS], b)
    for k, v in idx.items(): summarize(v, k)
    return pd.DataFrame(feats, index=df.index).replace([np.inf, -np.inf], np.nan)

def corrupt(df, rate, seed=0):
    """Randomly blank band cells to mimic the test set's missingness."""
    rng = np.random.RandomState(seed); d = df.copy()
    bc = [c for c in d.columns if c not in ("ID", "label")]
    arr = d[bc].values.astype(float)
    arr[rng.rand(*arr.shape) < rate] = np.nan
    d[bc] = arr
    return d

In [3]:
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score

train = load_raw(TRAIN_PATH)
y = train["label"].values
X = build_features(train)
def mk(): return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
            max_leaf_nodes=7, l2_regularization=1.0, random_state=42)

## Step 1 — why random CV lies twice here

Random folds are optimistic for the usual reason (same conditions in train & val) **and** a new one: they validate on *clean* data, but the real test is corrupted. Let's measure the clean random-CV number as the "fantasy" baseline.

In [4]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
oof = cross_val_predict(mk(), X, y, cv=cv, method="predict_proba")[:,1]
f1=f1_score(y,(oof>=.5).astype(int)); auc=roc_auc_score(y,oof)
print(f"[random CV, clean]  F1={f1:.4f} AUC={auc:.4f} blend={0.6*f1+0.4*auc:.4f}  (fantasy)")

[random CV, clean]  F1=0.9720 AUC=0.9956 blend=0.9814  (fantasy)


## Step 2 — feature-space clusters replace geography

In [5]:
clusters = KMeans(n_clusters=10, random_state=42, n_init=10).fit_predict(X.fillna(X.median()))
print("cluster sizes:", np.bincount(clusters))
gkf = GroupKFold(5)

cluster sizes: [182 417  48 156 143 175 129 156 102 313]


## Step 3 — the honest estimate: cluster holdout **+ corrupted validation**

For each fold we hold out a feature-cluster *and* blank ~60% of its cells to look like the real test. This is the number that should track the live leaderboard.

In [6]:
oof = np.zeros(len(y))
for tr, va in gkf.split(X, y, clusters):
    Xv = build_features(corrupt(train.iloc[va], rate=0.6, seed=99))   # corrupt validation
    m = mk().fit(X.iloc[tr], y[tr])
    oof[va] = m.predict_proba(Xv)[:,1]
f1=f1_score(y,(oof>=.5).astype(int)); auc=roc_auc_score(y,oof)
print(f"[cluster CV, corrupted val]  F1={f1:.4f} AUC={auc:.4f} blend={0.6*f1+0.4*auc:.4f}  (HONEST)")

[cluster CV, corrupted val]  F1=0.7703 AUC=0.9237 blend=0.8317  (HONEST)


**This honest number lands far below the clean fantasy** — and right in the range of the live leaderboard. That gap is the cost of the missing-data shift. Closing it (Stage 5) is the real work.

In [7]:
# reusable honest evaluator (the one later stages import)
def Xfull_fill(X): return X.fillna(X.median())

def honest_cv(make_estimator, train_df, fit_fn=None, n_clusters=10, n_splits=5,
              corrupt_rate=0.6, verbose=True, name=""):
    y = train_df["label"].values
    Xfull = build_features(train_df)
    clusters = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit_predict(Xfull_fill(Xfull))
    gkf = GroupKFold(n_splits); oof = np.zeros(len(y))
    for tr, va in gkf.split(Xfull, y, clusters):
        Xv = build_features(corrupt(train_df.iloc[va], corrupt_rate, seed=99))
        if fit_fn is None:
            m = make_estimator().fit(Xfull.iloc[tr], y[tr]); oof[va] = m.predict_proba(Xv)[:,1]
        else:
            oof[va] = fit_fn(train_df.iloc[tr], y[tr], Xv)
    f1=f1_score(y,(oof>=.5).astype(int)); auc=roc_auc_score(y,oof); bl=0.6*f1+0.4*auc
    if verbose: print(f"{name:28s} F1={f1:.4f} AUC={auc:.4f} blend={bl:.4f}")
    return f1, auc, bl

_ = honest_cv(mk, train, name="baseline HGB (honest)")

baseline HGB (honest)        F1=0.7703 AUC=0.9237 blend=0.8317


## Stage 3 takeaways
- No coordinates → **feature-space clustering** for holdout (honest-ish proxy for unseen conditions).
- **Corrupting the validation fold** to ~60% missing makes the estimate realistic — it drops to the leaderboard's range.
- Saved `honest_cv(...)`: every later experiment is judged by it.
- The clean-vs-corrupted gap **is** the missing-data challenge, quantified.

**Next (Stage 4):** baselines under this honest evaluator.